In [1]:
import os
print(os.getcwd())
print(os.listdir("../data/raw"))



/Users/casenanayake/Documents/olist-analytics/notebooks
['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [2]:
import glob, sqlite3
import pandas as pd

conn = sqlite3.connect("../olist.db")

for f in glob.glob("../data/raw/*.csv"):
    name = os.path.basename(f).replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df = pd.read_csv(f)
    df.to_sql(name, conn, if_exists="replace", index=False)
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

sellers: 3,095 rows, 4 columns
product_category_name_translation: 71 rows, 2 columns
orders: 99,441 rows, 8 columns
order_items: 112,650 rows, 7 columns
customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
products: 32,951 rows, 9 columns


In [3]:
tables = {}
for t in ["orders","customers","order_items","order_payments","order_reviews",
          "products","sellers","geolocation","product_category_name_translation"]:
    tables[t] = pd.read_sql(f"SELECT * FROM {t}", conn)

orders      = tables["orders"]
customers   = tables["customers"]
items       = tables["order_items"]
payments    = tables["order_payments"]
reviews     = tables["order_reviews"]
products    = tables["products"]
sellers     = tables["sellers"]
geo         = tables["geolocation"]
translation = tables["product_category_name_translation"]
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
for c in ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
          "order_delivered_customer_date","order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
for name, df in [("orders",orders),("products",products),("reviews",reviews),("items",items)]:
    print(name); print(df.isna().sum()[df.isna().sum()>0]); print()

print(orders["order_status"].value_counts())

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

items
Series([], dtype: int64)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
for name, df in tables.items():
    print(f"{name}: {df.duplicated().sum():,} duplicate rows")

print("Orders with more than one review:", reviews["order_id"].duplicated().sum())

orders: 0 duplicate rows
customers: 0 duplicate rows
order_items: 0 duplicate rows
order_payments: 0 duplicate rows
order_reviews: 0 duplicate rows
products: 0 duplicate rows
sellers: 0 duplicate rows
geolocation: 261,831 duplicate rows
product_category_name_translation: 0 duplicate rows
Orders with more than one review: 551


In [7]:
geo = geo.drop_duplicates()

# keep only the latest review per order
reviews = (reviews.sort_values("review_answer_timestamp")
                  .drop_duplicates("order_id", keep="last"))

# payments with no real type
payments = payments[payments["payment_type"] != "not_defined"]

print(len(geo), len(reviews), len(payments))

738332 98673 103883


In [8]:
# product categories
products["product_category_name"] = products["product_category_name"].str.strip().str.lower()
products = products.merge(translation, on="product_category_name", how="left")
products["category"] = (products["product_category_name_english"]
                        .fillna(products["product_category_name"])
                        .fillna("unknown"))

# cities and states
customers["customer_city"]  = customers["customer_city"].str.strip().str.lower()
customers["customer_state"] = customers["customer_state"].str.strip().str.upper()
sellers["seller_city"]  = sellers["seller_city"].str.strip().str.lower()
sellers["seller_state"] = sellers["seller_state"].str.strip().str.upper()

products["category"].value_counts().head(10)

category
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

In [9]:
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delay_days"]    = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days
orders["is_late"]       = orders["delay_days"] > 0
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

items["revenue"] = items["price"] + items["freight_value"]

orders.loc[orders["order_status"]=="delivered", ["delivery_days","delay_days","is_late"]].describe()

,delivery_days,delay_days
count,96470.000000,96470.000000
mean,12.093604,-11.875889
std,9.551380,10.182105
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [10]:
import os
os.makedirs("../data/clean", exist_ok=True)

clean = {"orders":orders, "customers":customers, "order_items":items,
         "order_payments":payments, "order_reviews":reviews,
         "products":products, "sellers":sellers, "geolocation":geo}

for name, df in clean.items():
    df.to_csv(f"../data/clean/{name}.csv", index=False)
    df.to_sql(f"{name}_clean", conn, if_exists="replace", index=False)
    print("saved", name, len(df))

saved orders 99441
saved customers 99441
saved order_items 112650
saved order_payments 103883
saved order_reviews 98673
saved products 32951
saved sellers 3095
saved geolocation 738332
